# VarChat — Ölçüm defteri (Google Colab)

Modelleri **aynı ortamda, aynı donmuş test setiyle** ölçer:

1. **`qwen2.5:7b`**: Ollama kütüphanesindeki ham model (yereldeki uygulamanın kullandığı)
2. **`qwen-taban`**: aynı ham model, ama eğitilmiş modelle **aynı dönüştürme hattından** geçmiş (birleştirme hariç her adım aynı)
3. **`varchat`** (ilk sürüm, 204 örnek) — sonuçları Drive'da duruyor, yeniden ölçülmez
4. **`varchat-v2`**: taban + yeni LoRA (~500 denetlenmiş örnek), kalıcı olarak birleştirilmiş

Eğitimin etkisi **`qwen-taban` ile eğitilmiş model arasındaki farktır**: ikisi aynı araçlarla, aynı 4-bit sıkıştırmayla (Q4_K_M) üretilir.
Ollama artık LoRA adaptörlerini kabul etmediği için (`LoRA adapters are no longer supported`) adaptör taban modele birleştirilir.

Her model için 56 girdinin özeti üretilir (`b06`), sonra her cümle kaynağına karşı yargıçla denetlenir (`b07`). Yargıç hep ham `qwen2.5:7b`.
Daha önce ölçülmüş modeller (Drive'daki dosyalar) saniyeler içinde geçilir; yalnız yeni model üretilip ölçülür.

**Kullanım:** *Runtime → Change runtime type → **L4*** (birleştirme ~16 GB RAM ister; T4 yetmez) → *Runtime → **Run all***.
Drive izin penceresi açılınca hesabını seçip onayla.

- Sonuçlar Drive'da **`varchat-olcum/sonuclar`**, model dosyaları **`varchat-gguf`** klasörüne yazılır.
  Oturum koparsa yeniden *Run all* yap: biten işler tekrarlanmaz.
- Süre: yeni modelin dosyasını üretmek ~30-45 dk + ölçümü ~30 dk. Tarayıcı sekmesini açık tut.

## Ayar: hangi eğitilmiş model?

`VARCHAT` Ollama'daki ve sonuç dosyalarındaki adıdır; `LORA_DIR` eğitim defterinin adaptörü kaydettiği Drive klasörüdür.
Ad her eğitimde FARKLI olmalı: aynı adla eski modelin sonuç dosyası "zaten yapılmış" sanılır ve yeni model ölçülmez.

In [ ]:
VARCHAT = "varchat-v2"
LORA_DIR = '/content/drive/MyDrive/varchat-lora-v2'

## 0) GPU, bellek ve disk

In [ ]:
import os, shutil
!nvidia-smi --query-gpu=name,memory.total --format=csv
RAM_GB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f"RAM: {RAM_GB:.0f} GB | boş disk: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

## 1) Projeyi getir + paketler

In [ ]:
import os
if not os.path.exists('/content/varchat'):
    !git clone -q https://github.com/atu-ce/varchat.git /content/varchat
else:
    # 'sonuclar' klasörü Drive'a bağlı (sembolik bağ) olduğu için 'git pull' takılabilir: kodu güncelle, sonuclar'a dokunma
    !cd /content/varchat && git fetch -q origin && git checkout -q origin/main -- . ':(exclude)degerlendirme/sonuclar'
%cd /content/varchat
!git log --oneline -1 origin/main
# Yereldeki sürümlerle aynı istemci; requests Colab'ın kendi sürümü kalır (Colab paketleri ona bağlı)
!pip install -q "ollama==0.6.2" "rapidfuzz==3.14.5"

## 2) Drive'a bağlan: sonuçlar kalıcı olsun

Projedeki `degerlendirme/sonuclar` klasörü Drive'daki `varchat-olcum/sonuclar` klasörüne bağlanır.
Doğru cevap listesi ve donmuş arama kaydı ilk seferde Drive'a kopyalanır; üretim ve sadakat dosyaları doğrudan Drive'a yazılır.

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

SONUC = '/content/drive/MyDrive/varchat-olcum/sonuclar'
GGUF_DRIVE = '/content/drive/MyDrive/varchat-gguf'        # üretilen model dosyalarının yedeği
os.makedirs(SONUC, exist_ok=True)
os.makedirs(GGUF_DRIVE, exist_ok=True)

yerel = '/content/varchat/degerlendirme/sonuclar'
if not os.path.islink(yerel):
    for ad in os.listdir(yerel):
        kaynak = os.path.join(yerel, ad)
        if os.path.isfile(kaynak) and not os.path.exists(os.path.join(SONUC, ad)):
            shutil.copy2(kaynak, SONUC)          # Drive'da olanın üstüne yazılmaz
    shutil.rmtree(yerel)
    os.symlink(SONUC, yerel)
print("sonuçlar ->", SONUC)
print(sorted(os.listdir(yerel)))

## 3) Ollama'yı kur, başlat, ham modeli indir

Yereldeki uygulamayla aynı model: `qwen2.5:7b` (4-bit, Q4_K_M). Ortam bilgisi (GPU, Ollama sürümü, model kimliği) tez için Drive'a yazılır.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests, json, datetime

def ollama_hazir():
    try:
        return requests.get('http://127.0.0.1:11434/api/version', timeout=2).json()
    except Exception:
        return None

if not ollama_hazir():
    subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        if ollama_hazir():
            break
print("Ollama:", ollama_hazir())
assert ollama_hazir(), "Ollama başlamadı; /content/ollama.log dosyasına bak"

In [ ]:
!ollama pull qwen2.5:7b
!ollama run qwen2.5:7b "Merhaba. Tek kelimeyle cevap ver."
ps_cikti = !ollama ps
print("\n".join(ps_cikti))
if not any("GPU" in s for s in ps_cikti[1:]):
    print("\nUYARI: model GPU'da değil. Runtime type GPU mu? Değilse değiştirip baştan Run all yap.")

gpu = !nvidia-smi --query-gpu=name --format=csv,noheader
liste = !ollama list
kod_surumu = !git -C /content/varchat log --oneline -1
ortam = {"tarih": datetime.datetime.now().isoformat(timespec="minutes"), "gpu": list(gpu), "ollama": ollama_hazir(),
         "modeller": list(liste), "kod": list(kod_surumu)}
json.dump(ortam, open(f"{SONUC}/ortam_{datetime.date.today().isoformat()}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(json.dumps(ortam, ensure_ascii=False, indent=1))

## 4) Hızlı kontrol: denetim testleri (model çağırmaz, birkaç saniye)

In [ ]:
!python degerlendirme/denetim_testi.py | tail -1

## 5) Ham model: `qwen2.5:7b`

Önce özetler üretilir, sonra her atıflı cümle kaynağına karşı yargılanır. Daha önce bittiyse saniyeler içinde geçer.

In [ ]:
!python -u degerlendirme/b06_toplu_uret.py qwen2.5:7b

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model qwen2.5:7b

## 6) Model dosyalarını üret: birleştir + GGUF + Q4_K_M

1. Hugging Face'ten taban model (`Qwen/Qwen2.5-7B-Instruct`, ~15 GB) indirilir.
2. **Eğitilmiş model (`VARCHAT`):** Drive'daki LoRA adaptörü tabana kalıcı olarak eklenir (birleştirme), sonuç GGUF'a çevrilir ve Q4_K_M'e sıkıştırılır.
3. **qwen-taban:** aynı taban, birleştirme OLMADAN, aynı araçlarla GGUF + Q4_K_M yapılır (adil karşılaştırma için).
4. Dosyalar Drive'a (`varchat-gguf/`) yedeklenir. Drive'da zaten olan dosya yeniden üretilmez (qwen-taban ilk koşudan hazır).

In [ ]:
import os, shutil, subprocess, sys
YEREL = {ad: f"/content/gguf/{ad}-Q4_K_M.gguf" for ad in ("qwen-taban", VARCHAT)}
HEDEF = {ad: f"{GGUF_DRIVE}/{ad}-Q4_K_M.gguf" for ad in YEREL}
os.makedirs('/content/gguf', exist_ok=True)
for ad in YEREL:
    if os.path.exists(HEDEF[ad]) and not os.path.exists(YEREL[ad]):
        print(f"{ad}: Drive'da hazır, kopyalanıyor...")
        shutil.copy(HEDEF[ad], YEREL[ad])
URET = [ad for ad in YEREL if not os.path.exists(YEREL[ad])]
print("üretilecek:", URET or "yok, ikisi de hazır")
if URET:
    if VARCHAT in URET:
        assert os.path.exists(f"{LORA_DIR}/adapter_model.safetensors"), \
            f"{LORA_DIR}/adapter_model.safetensors yok: önce fine_tune/colab_egitim.ipynb çalışmalı."
    assert RAM_GB >= 30, f"RAM {RAM_GB:.0f} GB: birleştirme için Runtime → Change runtime type → L4 (ya da A100) seç, baştan Run all yap."
    assert shutil.disk_usage('/content').free / 1e9 >= 60, "En az ~60 GB boş disk gerekir."

In [ ]:
if URET:
    # Eğitimdekiyle aynı sürümler (adaptör bu sürümlerle kaydedildi)
    !pip install -q "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0" sentencepiece "protobuf>=4.21,<5"
    # Colab'ın hazır kurulu eski torchao'su (0.10) peft 0.21'in birleştirme yolunu bozuyor ("incompatible version of torchao");
    # bu iş için gerekmez, kaldırılır (5 Ekim Colab hatası)
    !pip uninstall -y -q torchao
    !apt-get -qq install -y cmake libcurl4-openssl-dev > /dev/null
    if not os.path.exists('/content/llama.cpp'):
        !git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
    QUANTIZE = '/content/llama.cpp/build/bin/llama-quantize'
    if not os.path.exists(QUANTIZE):
        print("llama-quantize derleniyor (~3-5 dk)...")
        !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_SERVER=OFF > /content/cmake.log 2>&1 || tail -20 /content/cmake.log
        !cmake --build /content/llama.cpp/build --target llama-quantize -j 8 > /content/build.log 2>&1 || tail -30 /content/build.log
    assert os.path.exists(QUANTIZE), "llama-quantize derlenemedi; /content/build.log dosyasına bak"
    LLAMA_COMMIT = !git -C /content/llama.cpp log --oneline -1
    print("llama.cpp:", LLAMA_COMMIT[0])

In [ ]:
if URET:
    from huggingface_hub import snapshot_download
    TABAN_HF = snapshot_download("Qwen/Qwen2.5-7B-Instruct")
    print("taban model:", TABAN_HF)

if VARCHAT in URET and not os.path.exists('/content/varchat-hf/config.json'):
    import gc, torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    print("birleştiriliyor (CPU, ~10 dk)...")
    taban = AutoModelForCausalLM.from_pretrained(TABAN_HF, dtype=torch.bfloat16, device_map="cpu", low_cpu_mem_usage=True)
    birlesik = PeftModel.from_pretrained(taban, LORA_DIR).merge_and_unload()
    birlesik.save_pretrained('/content/varchat-hf', safe_serialization=True, max_shard_size="4GB")
    AutoTokenizer.from_pretrained(TABAN_HF).save_pretrained('/content/varchat-hf')
    if os.path.exists(f"{TABAN_HF}/generation_config.json"):
        shutil.copy(f"{TABAN_HF}/generation_config.json", '/content/varchat-hf/')
    del taban, birlesik
    gc.collect()
    print("birleştirildi -> /content/varchat-hf")

In [ ]:
def donustur(hf_dizin, f16, python):
    return subprocess.run([python, "/content/llama.cpp/convert_hf_to_gguf.py", hf_dizin, "--outtype", "f16", "--outfile", f16],
                          capture_output=True, text=True)

def ayri_ortam():
    # Dönüştürme betiği yeni transformers ile çalışmazsa: transformers<5 olan ayrı bir sanal ortam (Colab ortamı bozulmaz)
    py = "/content/donustur-venv/bin/python"
    if not os.path.exists(py):
        !python -m venv /content/donustur-venv || (pip install -q virtualenv && python -m virtualenv -q /content/donustur-venv)
        !/content/donustur-venv/bin/pip install -q torch --index-url https://download.pytorch.org/whl/cpu
        !/content/donustur-venv/bin/pip install -q "transformers>=4.45,<5" sentencepiece "protobuf>=4.21,<5" numpy safetensors pyyaml tqdm requests
    assert os.path.exists(py), "Ayrı dönüştürme ortamı kurulamadı (yukarıdaki çıktıyı Claude'a gönder)"
    return py

PYTHON = sys.executable
for ad in URET:
    hf = TABAN_HF if ad == "qwen-taban" else '/content/varchat-hf'
    f16 = f"/content/gguf/{ad}-f16.gguf"
    print(f"{ad}: GGUF'a çevriliyor...")
    r = donustur(hf, f16, PYTHON)
    if r.returncode != 0 and PYTHON == sys.executable:
        print("   dönüştürme bu ortamda başarısız, ayrı ortamda tekrar deneniyor:", r.stderr.strip().splitlines()[-1:] )
        PYTHON = ayri_ortam()
        r = donustur(hf, f16, PYTHON)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-3000:])
        raise SystemExit(f"{ad}: GGUF dönüştürme başarısız (çıktıyı Claude'a gönder)")
    print(f"{ad}: Q4_K_M'e sıkıştırılıyor...")
    r = subprocess.run([QUANTIZE, f16, YEREL[ad], "Q4_K_M"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise SystemExit(f"{ad}: Q4_K_M sıkıştırma başarısız (çıktıyı Claude'a gönder)")
    os.remove(f16)
    print(f"{ad}: hazır, {os.path.getsize(YEREL[ad]) / 1e9:.2f} GB")
    if ad == VARCHAT:
        shutil.rmtree('/content/varchat-hf', ignore_errors=True)
    bos = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
    if bos > 6:
        shutil.copy(YEREL[ad], HEDEF[ad])
        print(f"{ad}: Drive'a yedeklendi -> {HEDEF[ad]}")
    else:
        print(f"{ad}: Drive'da yer az ({bos:.1f} GB); yedeklenmedi (oturum koparsa yeniden üretilir)")
if URET:
    import hashlib
    bilgi = {"tarih": datetime.datetime.now().isoformat(timespec="minutes"), "llama_cpp": LLAMA_COMMIT[0],
             "donusturme_python": PYTHON, "dosyalar": {ad: {"boyut": os.path.getsize(y)} for ad, y in YEREL.items()}}
    json.dump(bilgi, open(f"{SONUC}/gguf_ortam_{datetime.date.today().isoformat()}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(json.dumps(bilgi, ensure_ascii=False, indent=1))

## 7) Model dosyalarını Ollama'ya yükle

Her iki dosya, ham `qwen2.5:7b` ile **aynı sohbet şablonu ve aynı ayarlarla** yüklenir (`ollama show --modelfile` ile alınır, yalnızca
`FROM` satırı değişir). Bu yöntem yerelde denendi: aynı dosyayla oluşturulan kopya, ham modelle harfi harfine aynı cevabı verdi.

In [ ]:
import re, ollama
ms = !ollama show qwen2.5:7b --modelfile
taban_mf = "\n".join(ms)
for ad in ("qwen-taban", VARCHAT):
    assert os.path.exists(YEREL[ad]), f"{YEREL[ad]} yok: 6. bölüm tamamlanmamış"
    with open(f"/content/Modelfile.{ad}", "w", newline="\n") as f:
        f.write(re.sub(r"(?m)^FROM .*$", f"FROM {YEREL[ad]}", taban_mf))
    !ollama create {ad} -f /content/Modelfile.{ad}
    try:
        ollama.show(ad)
    except ollama.ResponseError as e:
        raise SystemExit(f"{ad} Ollama'ya yüklenemedi: {e}. Yukarıdaki 'ollama create' çıktısını Claude'a gönder.")
    print(f"{ad}: Ollama'da hazır")

# Kısa kontrol: aynı soru, üç model (yalnızca göz kontrolü; sayısal ölçüm sonraki bölümlerde)
soru = [{"role": "system", "content": "Yanıtın DAİMA Türkçe olmalı."}, {"role": "user", "content": "BRAF V600E nedir? Bir cümleyle anlat."}]
for ad in ("qwen2.5:7b", "qwen-taban", VARCHAT):
    r = ollama.chat(model=ad, messages=soru, options={"temperature": 0, "seed": 42, "num_predict": 60, "num_ctx": 8192})
    print(f"{ad:12} {r['message']['content'][:160]!r}")

## 8) `qwen-taban` ölçümü (aynı hattan geçmiş taban)

In [ ]:
!python -u degerlendirme/b06_toplu_uret.py qwen-taban

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model qwen-taban

## 9) Eğitilmiş modelin (`VARCHAT`) ölçümü (taban + LoRA, birleştirilmiş)

Yargıç yine ham `qwen2.5:7b`.

In [ ]:
!python -u degerlendirme/b06_toplu_uret.py {VARCHAT}

In [ ]:
!python -u degerlendirme/b07_sadakat.py --model {VARCHAT}

## 10) Karşılaştırma tablosu

Katman başına: sadakat (kaynağın desteklediği iddia oranı, yüksek iyi), çelişki ve atıfsız cümle oranı (düşük iyi),
geçersiz atıf, dil çöküşü, kesilen cevap, süre. Eğitimin etkisi: **qwen-taban → eğitilmiş model**; veri düzeltmesinin etkisi:
**varchat → varchat-v2**. Tablo Drive'a `karsilastirma_<tarih>.csv` olarak da yazılır.

In [ ]:
import glob, pandas as pd

def son_ozet(model_ad):
    d = [x for x in glob.glob(f"{SONUC}/sadakat_ozet_{model_ad}_*.json") if "__yargic-" not in x]   # ana yargıç (7B) dosyaları
    return json.load(open(max(d, key=os.path.getmtime), encoding="utf-8")) if d else None

modeller = {"qwen2.5:7b (Ollama)": son_ozet("qwen2.5-7b"), "qwen-taban": son_ozet("qwen-taban"),
            "varchat (v1, 204 örnek)": son_ozet("varchat"), f"{VARCHAT} (taban+LoRA)": son_ozet(VARCHAT.replace(":", "-"))}
OLCULER = [("sadakat", "Sadakat ↑"), ("celiski_orani", "Çelişki oranı ↓"), ("atifsiz_orani", "Atıfsız cümle oranı ↓"),
           ("gecersiz_atif", "Geçersiz atıf ↓"), ("cjk_bozuk", "Dil çöküşü ↓"), ("kesilen", "Kesilen cevap ↓"),
           ("uretim_hatasi", "Üretim hatası (model döngüsü) ↓"), ("iddia", "Yargılanan iddia"),
           ("belirsiz", "Belirsiz yargı (sadakate girmez)"), ("yargic_hatasi", "  bunun yargıç hatası olanı"),
           ("ort_sure_s", "Ortalama süre (sn)")]
satirlar = []
katmanlar = sorted({k for o in modeller.values() if o for k in o["tablo"]}, key=lambda k: (k == "TÜMÜ", k))
for k in katmanlar:
    for anahtar, ad in OLCULER:
        satirlar.append({"katman": k, "ölçü": ad,
                         **{m: (o["tablo"].get(k, {}).get(anahtar) if o else None) for m, o in modeller.items()}})
tablo = pd.DataFrame(satirlar)
tablo.to_csv(f"{SONUC}/karsilastirma_{datetime.date.today().isoformat()}.csv", index=False, encoding="utf-8-sig")
for m, o in modeller.items():
    print(f"{m}: {o['uretim'] if o else 'henüz ölçülmedi'}")
pd.set_option("display.max_rows", 200)
display(tablo)

### Eşleştirilmiş karşılaştırma (aynı 56 girdi; %95 güven aralığı ve işaret testi)

In [ ]:
!python degerlendirme/b08_karsilastir.py --ciftler "qwen-taban>{VARCHAT},varchat>{VARCHAT},qwen-taban>varchat"

## 11) Sonuçları bilgisayarına al

Drive'daki **`varchat-olcum/sonuclar`** klasöründen şu dosyaları indir:
`uretim_*.jsonl`, `sadakat_*.jsonl`, `sadakat_ozet_*.json`, `insan_kontrol_*.csv`, `karsilastirma_*.csv`, `ortam_*.json`, `gguf_ortam_*.json`.

`insan_kontrol_*.csv`: yargıcın kararlarından rastgele 30 iddia. Son sütunu sen elle doldurursun; yargıcın ne kadar güvenilir
olduğunu (insan-yargıç uyumu) bununla ölçeriz. Model dosyalarını (`varchat-gguf/*.gguf`, ~4,7 GB) indirmen gerekmez.